# 15 · 第三阶段 G 的前提查验

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → 第二阶段 F ✅（14a–14f + 附录 14x）从一个分数到两个能量
                 ↓
      第三阶段 G：从"评价已有的"到"提出没有的"
      └─ 15 前提查验  ← 你在这里（章节定稿前的闸门）
```

## 为什么要先有这一章

第三阶段的想法是：两个能量不只能解释已有的突变，还能**指挥改造**——生成器提方案，两个能量做选择，稳定性往上、功能不动。

但这个想法的初稿压在两个假设上，而**假设没查就是空想**。这一章把它们查掉：

| 假设 | 为什么必须查 |
|---|---|
| 湖里有大量两个能量的打分，可以当下游模型的训练集 | 如果那些蛋白是考试蛋白，喂给生成器就等于把考卷喂进去 |
| 多突变数据够训一个多突变打分器 | 设计产出一次改多个位置，只在单突变上验证过的打分器不能信 |

**这一章的结论是负面的**，而且负面结论照写——08b、08c 都是这么写的。

In [ ]:
import os
import pickle
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from scipy import stats

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
WORK = ROOT / "data/stability"   # 14a 的本机工作副本，不进 Git

CSV = WORK / "Tsuboyama2023_Dataset2_Dataset3_20230416.csv"
SPLITS = WORK / "mega_splits.pkl"
SINGLE = re.compile(r"^([A-Z])(\d+)([A-Z])$")
print(CSV.exists(), SPLITS.exists())

---

## 第 ① 步 · 查验一：两个能量覆盖了多少蛋白

### 在干什么
初稿想拿"湖里已有的两个能量打分"当条件生成器的训练集。先确认那批打分到底是什么、覆盖哪些蛋白。

### 怎么查
不用跑代码——14d 的记账和 README 已经写明了，只要把两个数字放在一起看。

| | 初稿假设 | 实际 |
|---|---|---|
| 湖里那 2.17 亿条打分 | 两个能量 | **09c 的旧模型、单个分数** |
| 两个能量重打的范围 | 接近全蛋白组 | **118 个蛋白 / 103,744 个位置 / 197 万个替换**（14d） |
| 那 118 个蛋白是谁 | 任意一批 | **108 个是锁死的考试蛋白** + KRAS、ABL1、SHP2 |

### 结论

**湖里的两个能量就是考试集。**

任何拿它当训练数据的下游模型都等于把考卷喂进去，之后在考试蛋白上的全部回溯验证作废。初稿里"用湖里已有打分训条件生成器"这条路堵死——不是打补丁的问题，是换底座。

替代底座只能是 **train split 蛋白的实测值**。那就要先知道实测的多突变有多少，于是有了第 ② 步。

---

## 第 ② 步 · 查验二：多突变数据的真实库存

### 在干什么
`stability.megascale` 在复现 ThermoMPNN 的过滤链时，有一步**同时**丢掉了插入、缺失和多突变（607,839 → 391,090）。那一步丢掉的多突变，就是第三阶段要用的料。先数它。

### 怎么保证数得对
**先自检过滤链**：三个已发表的数字必须一个不差地复现出来，否则说明数据或过滤逻辑变了，后面的计数都不可信。

In [ ]:
# 读一次，后面各步复用（697 MB，约半分钟）
raw = pd.read_csv(CSV, usecols=["ddG_ML", "mut_type", "WT_name", "aa_seq"], low_memory=False)

print(f"{'文件行数':<24}{len(raw):>10,}   已发表 776,298")
withddg = raw[raw.ddG_ML != "-"]
print(f"{'有可靠 ddG':<22}{len(withddg):>10,}   已发表 607,839")

indel = withddg.mut_type.str.contains("ins|del", regex=True)
multi = ~indel & withddg.mut_type.str.contains(":")
single = ~indel & ~multi & (withddg.mut_type != "wt")
wt = withddg.mut_type == "wt"

print(f"{'  插入 / 缺失':<21}{int(indel.sum()):>10,}")
print(f"{'  多突变':<23}{int(multi.sum()):>10,}")
print(f"{'  单突变':<23}{int(single.sum()):>10,}")
print(f"{'  野生型':<23}{int(wt.sum()):>10,}")
print(f"{'  单突变 + 野生型':<19}{int(single.sum() + wt.sum()):>10,}   已发表 391,090")

三个数字全部对上，计数可信。现在看多突变池按已发表 split 怎么分布。

In [ ]:
class _NoCodeUnpickler(pickle.Unpickler):
    """只允许 numpy 的数组机制：pickle 是程序不是数据文件，不给它执行任何东西的机会。"""

    ALLOWED = {
        ("numpy.core.multiarray", "_reconstruct"),
        ("numpy._core.multiarray", "_reconstruct"),
        ("numpy", "ndarray"),
        ("numpy", "dtype"),
    }

    def find_class(self, module, name):
        if (module, name) not in self.ALLOWED:
            raise pickle.UnpicklingError(f"refused to import {module}.{name}")
        return getattr(__import__(module, fromlist=[name]), name)


with open(SPLITS, "rb") as f:
    splits = {k: [str(x) for x in v] for k, v in _NoCodeUnpickler(f).load().items() if k in ("train", "val", "test")}
where = {name: split for split, names in splits.items() for name in names}

m = withddg[multi].copy()
m["k"] = m.mut_type.str.count(":") + 1
m["split"] = m.WT_name.map(where)

print("按突变个数 k:", m.k.value_counts().sort_index().to_dict())
print()
summary = m.groupby(m.split.fillna("不在已发表 split 内")).agg(行数=("mut_type", "size"), 结构域=("WT_name", "nunique"))
summary.loc["可用于训练 (train+val)"] = [
    int(m.split.isin(["train", "val"]).sum()),
    m[m.split.isin(["train", "val"])].WT_name.nunique(),
]
print(summary.to_string())
print()
for name, names in splits.items():
    print(f"  {name:<6}{m[m.WT_name.isin(names)].WT_name.nunique():>4} / {len(set(names)):>4} 个结构域有多突变")

### 第 ① 步的结论：池子够大，但有一个硬边界

**159,051 行多突变，153 个结构域**；可用于训练 136,970 行 / 133 个结构域，留出 22,081 行 / 20 个结构域当考卷。按结构域切分和 14a 完全一致，现有的防泄漏框架直接套用。

**但全部是 k=2，一个三突变及以上都没有。**

所以这批数据能验证"单突变 → 双突变"，**验证不了**"双突变 → 十几二十个突变"。设计产出要么限制在少量突变，要么必须标注成外推、没有实测支撑。这条边界不许含糊。

还要再砍一刀：14a 查出 56 个结构域和锁死考试蛋白同源，其中 **52 个在 Tsuboyama 训练侧**。这 133 个可用结构域要用同一条同源规则过一遍，预计掉 15–20%。

---

## 第 ② 步 · 查验三：头部空间有多大

### 在干什么
"池子够大"不等于"值得做一章"。15a 的前提是**需要一个模型**来预测多突变能量。这个前提只有在**免费的替代方案不够用**时才成立。

免费的替代方案是：把两个组分单突变的**实测**能量直接相加。

### 为什么这个检验不会泄漏
全程只用实测值，不碰任何模型。

### 先清理数据
双突变写成 `T9Q:N23Q`。注意有 `T9Q:N23N` 这种——第二个改动是**同义的**（N→N），那其实是单突变，必须滤掉。

In [ ]:
d = m.copy()
d["ddg"] = pd.to_numeric(d.ddG_ML)
parts = d.mut_type.str.split(":", expand=True)
for i in (0, 1):
    p = parts[i].str.extract(SINGLE)
    d[[f"wt{i}", f"pos{i}", f"mut{i}"]] = p
    d[f"pos{i}"] = pd.to_numeric(d[f"pos{i}"])
d = d.dropna(subset=["wt0", "wt1"])

synonymous = (d.wt0 == d.mut0) | (d.wt1 == d.mut1)
print(f"池子里的双突变          {len(d):>9,}")
print(f"  有同义组分 (X->X)   {int(synonymous.sum()):>9,}   <- 其实是单突变，滤掉")
d = d[~synonymous].copy()
print(f"  真正的双替换        {len(d):>9,}")

# 单突变的实测值，用来相加
s = withddg[single].copy()
s = s[s.mut_type.str.match(SINGLE)]
singles = s.assign(ddg_single=pd.to_numeric(s.ddG_ML)).set_index(["WT_name", "mut_type"]).ddg_single

for i in (0, 1):
    name = d[f"wt{i}"] + d[f"pos{i}"].astype(str) + d[f"mut{i}"]
    d[f"ddg{i}"] = singles.reindex(pd.MultiIndex.from_arrays([d.WT_name, name])).values

both = d.dropna(subset=["ddg0", "ddg1"]).copy()
both["additive"] = both.ddg0 + both.ddg1
both["residual"] = both.ddg - both.additive
both["separation"] = (both.pos1 - both.pos0).abs()
print(f"两个组分都有实测值的   {len(both):>9,}   ({len(both) / len(d):.1%})，{both.WT_name.nunique()} 个结构域")

In [ ]:
def report(frame, label):
    x, y = frame.additive.to_numpy(), frame.ddg.to_numpy()
    ss_res = float(((y - x) ** 2).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum())
    return {
        "窗口": label,
        "n": len(frame),
        "结构域": frame.WT_name.nunique(),
        "Pearson": round(stats.pearsonr(x, y)[0], 3),
        "Spearman": round(stats.spearmanr(x, y)[0], 3),
        "R2": round(1 - ss_res / ss_tot, 3),
        "RMSE": round(np.sqrt(ss_res / len(frame)), 3),
        "偏差": round(frame.residual.mean(), 3),
        "残差SD": round(frame.residual.std(), 3),
    }


rows = [report(both, "全部")] + [
    report(both[both.split == n], f"已发表 {n} split") for n in ("train", "val", "test")
]
print(pd.DataFrame(rows).to_string(index=False))

### 一个矛盾的结果，必须解释清楚

可加性**排序很好**（Pearson / Spearman 都是 0.81），但 **R² = −0.42**，比直接预测均值还差。

两者能同时成立，只有一个原因：**系统性偏差**。残差均值 **+0.894 kcal/mol**，不是散点，是整体偏移。

偏移从哪来？两个候选解释，而区分它们决定 15a 到底是什么章：

1. **真实的上位效应**——两个突变互相影响。如果是这个，残差应该在**两个位置靠得近**的时候更大（接触才能耦合）。
2. **测量被压顶**——实验测不到太失稳的蛋白。如果是这个，残差应该随**预测的严重程度**单调增长，而和位置间距无关。

In [ ]:
print("=== 残差 vs 位置间距（真上位效应的签名）===")
bins = pd.cut(both.separation, [0, 4, 8, 16, 32, 10_000], labels=["1-4", "5-8", "9-16", "17-32", ">32"])
print(both.groupby(bins, observed=True).residual.agg(["size", "mean", "std"]).round(3).to_string())

print()
print("=== 残差 vs 可加预测值（压顶的签名）===")
print(f"相关系数 {stats.pearsonr(both.additive, both.residual)[0]:+.3f}")
print()
print(
    both.groupby(pd.qcut(both.additive, 8), observed=True)
    .agg(n=("ddg", "size"), 可加预测=("additive", "mean"), 实测=("ddg", "mean"), 残差=("residual", "mean"))
    .round(3)
    .to_string()
)

In [ ]:
print("=== 实测 ddG 有没有硬地板 ===")
for q in (0, 0.1, 0.5, 1, 2, 5):
    print(f"  第 {q:>3}% 分位   {both.ddg.quantile(q / 100):>7.3f}")
print(f"  双突变最低 {both.ddg.min():.2f}   最高 {both.ddg.max():.2f}")
print(f"  单突变最低 {both.ddg0.min():.2f}   最高 {both.ddg0.max():.2f}   <- 同一个地板")

deep = both[both.additive < -5]
mid = both[(both.additive > -1) & (both.additive < 1)]
print()
print(f"  可加预测 < -5 的 {len(deep):,} 行：预测均值 {deep.additive.mean():.2f}，实测均值 {deep.ddg.mean():.2f}，残差 {deep.residual.mean():+.2f}")
print(f"  对照（可加预测 -1~1）：实测均值 {mid.ddg.mean():.2f}，残差 {mid.residual.mean():+.2f}")

### 压顶确认，上位效应没有签名

- 残差 vs 可加预测值相关 **−0.72**，单调：预测 −0.55 时残差 +0.11，预测 −5.55 时残差 **+2.23**。
- 实测 ddG 的地板在 **−5.69**，单突变地板 −5.52，**同一个地板**——这是实验的量程，不是蛋白的性质。
- 可加预测 −5.98 的那批，实测平均只有 **−3.52**。
- 而残差在"两个位置相距 >32 残基"时还是 **0.98**，和靠得很近时（0.67）差不多。那种距离基本不接触，**真实的结构性上位效应应该消失，它没有**。

**结论：那 0.9 的"非可加性"绝大部分是测量截断，不是生物学。**

两个失稳突变相加经常预测到 −6 ~ −10 kcal/mol，超出这个实验能测的范围，实测就饱和在 −5 附近。

### 那么把超量程的去掉，剩下的干净窗口有多大

In [ ]:
rows = []
for t in (-10, -5, -4, -3, -2.5, -2, -1.5):
    r = report(both[both.additive > t], f"可加预测 > {t}")
    r["test侧"] = int((both[both.additive > t].split == "test").sum())
    rows.append(r)
print(pd.DataFrame(rows)[["窗口", "n", "结构域", "test侧", "Pearson", "R2", "RMSE", "偏差", "残差SD"]].to_string(index=False))

print()
print("=== 干净窗口里，残差还跟位置间距有关吗 ===")
c = both[(both.additive > -2.5) & (both.ddg > -4.2)]
print(c.groupby(pd.cut(c.separation, [0, 4, 8, 16, 32, 10_000], labels=["1-4", "5-8", "9-16", "17-32", ">32"]),
                observed=True).residual.agg(["size", "mean", "std"]).round(3).to_string())

窗口收紧时发生了两件事，都对 15a 不利：

- **偏差变小但从不消失**（0.894 → 0.200），而且 R² 最高只到 **0.175**；
- **相关系数反而下降**（0.807 → 0.578）。因为收紧窗口把动态范围一起切掉了——原来那个 0.81 的相关，大半是失稳尾巴这些**容易的样本**撑起来的。

而且干净窗口里残差仍然**和位置间距无关**（1-4 是 0.348，>32 是 0.376）。所以剩下的偏差还是系统性的压缩，不是接触耦合。

### 最后一件事，也是对第三阶段最关键的

压顶发生在**失稳端**，而稳定化改造在**另一端**。所以要单独看稳定端——那才是"提高稳定性"这件事的靶子。

In [ ]:
print("=== 稳定端：可加预测 > 0 ===")
for lo in (0, 0.3, 0.5):
    f = both[both.additive > lo]
    x, y = f.additive.to_numpy(), f.ddg.to_numpy()
    print(f"  可加预测>{lo}: n={len(f):,}  结构域={f.WT_name.nunique()}  test侧={int((f.split == 'test').sum())}  "
          f"Pearson={stats.pearsonr(x, y)[0]:.3f}  偏差={f.residual.mean():+.3f}")

print()
print("=== 实测确实更稳定的双突变有多少 ===")
for t in (0, 0.5, 1.0):
    f = both[both.ddg > t]
    print(f"  实测 ddG > +{t}: {len(f):>7,} 行, {f.WT_name.nunique():>3} 个结构域, test 侧 {int((f.split == 'test').sum()):>5} 行")

print()
uniq = both[["WT_name", "wt0", "pos0", "mut0", "ddg0"]].drop_duplicates()
print(f"=== 单突变的候选空间：去重 {len(uniq):,} 个，其中实测 ddG>0 的 {int((uniq.ddg0 > 0).sum()):,} 个，>+0.5 的 {int((uniq.ddg0 > 0.5).sum()):,} 个 ===")

up = both[(both.ddg0 > 0) & (both.ddg1 > 0)]
x, y = up.additive.to_numpy(), up.ddg.to_numpy()
print()
print(f"=== 两个组分都稳定化的双突变：n={len(up):,} ===")
print(f"  Pearson={stats.pearsonr(x, y)[0]:.3f}  偏差={up.residual.mean():+.3f}  RMSE={np.sqrt(((y - x) ** 2).mean()):.3f}")
print(f"  实测真的比两者单独都更稳定的比例：{(up.ddg > up[['ddg0', 'ddg1']].max(axis=1)).mean():.1%}  <- 接近抛硬币")

---

## 结论：第三阶段的初稿不成立，必须改

三条，按严重程度排：

### 1 · 湖里的两个能量就是考试集

"用已有打分训条件生成器"堵死。替代底座只能是 train split 蛋白的实测值。

### 2 · 多突变池看着大，实际被测量截断主导

159,051 行里真正的双替换 142,207 行，两个组分都有实测值的 131,143 行。但可加性的"失败"（R² −0.42）**绝大部分是实验量程压顶**，不是上位效应——残差和位置间距无关，和预测严重程度强相关（−0.72），而且双突变与单突变共用同一个 −5.5 地板。

把超量程的切掉，R² 最高只到 0.175，相关系数反而掉到 0.58。

**所以 15a 不能是"我们学会了上位效应"这一章。** 它能成立的版本是**测量模型**那一章：把压缩显式建模（14c 的读数层 + 死亡水平就是这套机制），说明扣掉压缩之后能量是相加的。这是个方法学上的负面结果，但它是真结果。

### 3 · 稳定端的数据撑不起回溯验证——这条最致命

我原来提的设计章节基准是"在留出结构域上算 precision@k"。查完之后它不成立：

| | 数量 |
|---|---|
| 去重单突变里实测稳定化的（ddG>0） | **350 / 6,875**（5%） |
| 其中 > +0.5 kcal/mol | **79** |
| 实测 ddG > +0.5 的双突变，**test 侧** | **173 行** |
| 实测 ddG > +1.0 的双突变，**test 侧** | **20 行** |

而且稳定端的可加性本身就差：两个组分都稳定化时 Pearson 只有 **0.525**，偏差翻成 **−0.21**（叠加起来**不如**两者之和，收益递减），**实测真的比两者单独都好的只有 53.7%——抛硬币**。

所以"把稳定化突变叠起来"这个设计循环的核心前提，在这批数据的 k=2 上**得不到支持**；而留出侧 20–173 行、散在 20–43 个结构域上，也**没有统计功效**支撑任何 precision@k 的说法。

---

## 新增的规矩（第三阶段起生效）

- **模型自己的输出不得作为下游模型的训练数据**，除非来源蛋白全部在 train split。14d 的位点地图、09c 的全蛋白组打分都踩这条线。
- **外部预训练模型算一个独立的泄漏面**：反向折叠模型训在 PDB 上（含考试蛋白结构），ESM-2 训在 UniRef 上（含考试蛋白序列）。现有 split 保护的是打分器，不是生成器。每用一个就要给出"只用它、不用我们的模型"的消融，**而且消融就是过关标准**。
- **多突变的留出级别要写明**：是"没见过的蛋白"还是"见过的蛋白、没见过的组合"。

---

## 不换题目：否掉的是基准，不是生成式

上面第 3 条容易被读成"生成式做不了"，**那是错的**，写清楚免得以后自己看错。

否掉的是**一个特定的验证设计**——在 Tsuboyama 留出结构域上用 precision@k 验稳定化改造。它死于稳定化突变太稀、留出侧太薄。但生成器本身（反向折叠、蛋白语言模型）这一章根本没碰到，而且**功效在别处已经有了**：

- **14x 的靶场**：128 组实验、92 个考试蛋白、22 万个突变，已经证明过引导挑选在 top 1% 里找到 7 个 vs 随机 4 个（91% 的实验更好，p = 3e-20）。把"从所有可能的单突变里挑"换成"生成器提候选、两个能量来挑"，评法和功效都照用。
- **ddPCA 的双轴**：同一个文库同时密集测了丰度和结合，正好是"稳定不掉、功能不丢"的现成考卷。KRAS 在 **test 侧**。
- **生成式自己的标准指标**：序列恢复率、重折叠自洽 RMSD、活性位点残基保留率，全部可算，不需要任何稳定化实测集。

**这一章的真实产出是一条精确的边界，不是叫停**：所有设计评估只在有实测的地方做，k≥3 的主张一律标成外推。

章节怎么切见 [`docs/plan.md`](../../docs/plan.md) 第 10 节。

## 下一步

查 FireProtDB（人工整理的实验验证稳定化突变）和已发表稳定化 campaign，看能不能给"功能保持的设计"那一章再加一个独立基准。**这是加分项，不是闸门**——生成器对照和设计循环那两章的过关标准已经不依赖它。

## 记账

这一章没有训练模型，所有数字由本 notebook 自己从 `data/stability` 重算。**MLflow 还没记**——当前环境 AWS 凭证不可用（`aws sts get-caller-identity` 报 NoCredentials），等凭证恢复后补记这几个计数。
